# Etapa 3.0 — Setup do experimento `reduzido`

Configura o ambiente MLflow para a Etapa 3 (SHAP + redução de inputs) e valida que os artefatos da Etapa 2 estão acessíveis.

**Checklist (PLANO_3.md §3.0):**
1. Reutilizar URI de tracking da Etapa 2.
2. Criar (ou reutilizar) o experimento `reduzido`.
3. Verificar acessibilidade dos 15 baselines via `mlflow.search_runs` e via filesystem.
4. Criar a estrutura de pastas em `ARTEFATOS/ETAPA_3/`.
5. Testar a função utilitária `load_baseline_model(modelo, output)`.

## 1. Imports e configuração de tracking

In [1]:
import pathlib
import pickle

import mlflow
from packaging.version import Version

print(f"MLflow version: {mlflow.__version__}")
assert Version(mlflow.__version__) >= Version("2.0"), "MLflow deve ser >= 2.x"

PROJECT_ROOT = pathlib.Path("/Users/lorenzoferreira/Documents/UFRGS/TCC_SBO")
TRACKING_URI = f"file:///{PROJECT_ROOT}/mlruns"

mlflow.set_tracking_uri(TRACKING_URI)
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

MLflow version: 3.11.1
Tracking URI: file:////Users/lorenzoferreira/Documents/UFRGS/TCC_SBO/mlruns


## 2. Criar/reutilizar o experimento `reduzido`

`mlflow.set_experiment` cria o experimento se não existir e o reutiliza caso já exista — operação idempotente.

In [2]:
experiment = mlflow.set_experiment("reduzido")
print(f"Experimento ID : {experiment.experiment_id}")
print(f"Experimento    : {experiment.name}")
print(f"Artefato root  : {experiment.artifact_location}")

Experimento ID : 834566610993577604
Experimento    : reduzido
Artefato root  : file:////Users/lorenzoferreira/Documents/UFRGS/TCC_SBO/mlruns/834566610993577604


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


## 3. Verificar acessibilidade dos baselines (Etapa 2)

Esperado: 15 combinações únicas `(modelo, output)` no experimento `baseline` — 5 arquiteturas × 3 outputs.

> O experimento pode conter mais de 15 runs por causa de re-execuções. O que importa é que existam runs `FINISHED` para todas as 15 combinações e que os artefatos correspondentes estejam no disco.

In [3]:
OUTPUTS = ["ET", "M_CH3OH", "x_CH3OH"]
MODELS  = ["SVR", "DT", "RF", "XGBoost", "ANN"]
EXPECTED_COMBOS = {(m, o) for m in MODELS for o in OUTPUTS}

runs = mlflow.search_runs(experiment_names=["baseline"])
print(f"Total de runs no experimento 'baseline': {len(runs)}")

finished = runs[runs["status"] == "FINISHED"].copy()
combos_disponiveis = set(zip(finished["tags.model"], finished["tags.output"]))
faltando = EXPECTED_COMBOS - combos_disponiveis
extras   = combos_disponiveis - EXPECTED_COMBOS

print(f"Combinações (modelo, output) com run FINISHED: {len(combos_disponiveis)} / 15")
if faltando:
    print(f"AVISO — combinações ausentes no MLflow: {sorted(faltando)}")
if extras:
    print(f"AVISO — combinações inesperadas no MLflow: {sorted(extras)}")

assert not faltando, "Há combinações (modelo, output) sem run FINISHED no baseline."

Total de runs no experimento 'baseline': 18
Combinações (modelo, output) com run FINISHED: 15 / 15


## 4. Função utilitária `load_baseline_model(modelo, output)`

Carrega o modelo serializado da Etapa 2 a partir do filesystem (não do MLflow), pois é a fonte autoritativa segundo `CONVENCOES_MLFLOW.md` e evita ambiguidade entre re-execuções.

Convenção de paths (decisão da Etapa 2):
- sklearn (SVR, DT, RF, XGBoost): `ARTEFATOS/ETAPA_2/<modelo>/<output>/model.pkl`
- ANN: `ARTEFATOS/ETAPA_2/ANN/<output>/model_v2.keras` (versão final usada na comparação 2.3)

In [4]:
ETAPA2_ROOT = PROJECT_ROOT / "ARTEFATOS" / "ETAPA_2"

SKLEARN_MODELS = {"SVR", "DT", "RF", "XGBoost"}
ANN_ARTIFACT_NAME = "model_v2.keras"


def baseline_artifact_path(modelo: str, output: str) -> pathlib.Path:
    """Retorna o caminho do artefato serializado do baseline (modelo, output)."""
    if modelo not in MODELS:
        raise ValueError(f"modelo desconhecido: {modelo!r}; esperado um de {MODELS}")
    if output not in OUTPUTS:
        raise ValueError(f"output desconhecido: {output!r}; esperado um de {OUTPUTS}")
    if modelo == "ANN":
        return ETAPA2_ROOT / "ANN" / output / ANN_ARTIFACT_NAME
    return ETAPA2_ROOT / modelo / output / "model.pkl"


def load_baseline_model(modelo: str, output: str):
    """Carrega o modelo baseline (modelo, output) da Etapa 2."""
    path = baseline_artifact_path(modelo, output)
    if not path.exists():
        raise FileNotFoundError(f"Artefato não encontrado: {path}")
    if modelo == "ANN":
        from tensorflow import keras  # import lazy — evita custo quando só sklearn é usado
        return keras.models.load_model(path)
    with open(path, "rb") as f:
        return pickle.load(f)

## 5. Verificar artefatos no filesystem e testar `load_baseline_model`

In [5]:
missing_files = []
for modelo in MODELS:
    for output in OUTPUTS:
        path = baseline_artifact_path(modelo, output)
        if not path.exists():
            missing_files.append(str(path))

if missing_files:
    print("FALHA — artefatos ausentes:")
    for p in missing_files:
        print(f"  - {p}")
else:
    print("OK — 15/15 artefatos de modelo presentes em ARTEFATOS/ETAPA_2/.")

assert not missing_files, "Há artefatos de baseline ausentes; rever Etapa 2 antes de prosseguir."

OK — 15/15 artefatos de modelo presentes em ARTEFATOS/ETAPA_2/.


In [6]:
# Teste funcional: carregar XGBoost × ET (caso citado na validação do PLANO_3.md §3.0)
xgb_et = load_baseline_model("XGBoost", "ET")
print(f"XGBoost × ET → {type(xgb_et).__module__}.{type(xgb_et).__name__}")

# Predição rápida em uma amostra do test set para confirmar que o modelo é utilizável
import numpy as np
X_test = np.load(PROJECT_ROOT / "ARTEFATOS" / "ETAPA_0" / "processed" / "X_test.npy")
y_hat  = xgb_et.predict(X_test[:5])
print(f"Predição em 5 amostras (espaço normalizado): {y_hat}")

XGBoost × ET → xgboost.sklearn.XGBRegressor
Predição em 5 amostras (espaço normalizado): [0.5353207  0.42542866 0.05952036 0.34903944 0.18479936]


## 6. Estrutura de pastas para artefatos da Etapa 3

Conforme PLANO_3.md §Organização de Arquivos.

In [7]:
ETAPA3_ROOT = PROJECT_ROOT / "ARTEFATOS" / "ETAPA_3"

subdirs = [
    ETAPA3_ROOT / "shap",
    ETAPA3_ROOT / "selecao",
    ETAPA3_ROOT / "equivalencia",
]

# reduzido/<modelo>/<output>/k<k>/
for modelo in MODELS:
    for output in OUTPUTS:
        for k in (4, 5, 6):
            subdirs.append(ETAPA3_ROOT / "reduzido" / modelo / output / f"k{k}")

for d in subdirs:
    d.mkdir(parents=True, exist_ok=True)

print(f"OK — {len(subdirs)} subdiretórios garantidos em {ETAPA3_ROOT}.")

OK — 48 subdiretórios garantidos em /Users/lorenzoferreira/Documents/UFRGS/TCC_SBO/ARTEFATOS/ETAPA_3.


## 7. Validação final

- Experimento `reduzido` existe e está vazio (nenhuma run própria ainda).
- As 15 combinações `(modelo, output)` do baseline têm run `FINISHED` e artefato no disco.
- `load_baseline_model` retorna um objeto utilizável.

In [8]:
runs_reduzido = mlflow.search_runs(experiment_names=["reduzido"])
n_reduzido = len(runs_reduzido)

print(f"Runs em 'reduzido': {n_reduzido}")
if n_reduzido == 0:
    print("OK — experimento vazio. Pronto para Etapa 3.1.")
else:
    print(f"AVISO — {n_reduzido} run(s) já existente(s). Verificar antes de prosseguir.")

print(f"\nMLflow UI: mlflow ui --backend-store-uri {TRACKING_URI}")

Runs em 'reduzido': 0
OK — experimento vazio. Pronto para Etapa 3.1.

MLflow UI: mlflow ui --backend-store-uri file:////Users/lorenzoferreira/Documents/UFRGS/TCC_SBO/mlruns
